In [ ]:
import openml
import pandas as pd
import os
from openai import OpenAI
import json
from tqdm import tqdm
import time
from itertools import islice
from collections import Counter
from pathlib import Path
client = OpenAI(api_key="") # insert openai api key for tokens

### Define project root and create the flowey output directory.

In [ ]:
ROOT = Path.cwd().parent

flows_dir = ROOT / "flows"
flows_dir.mkdir(exist_ok=True)

### Download all OpenML flows and save flow IDs and names to a CSV.

In [ ]:
flows = openml.flows.list_flows(output_format="dataframe")
flows = flows[["id", "name"]]

flows.to_csv(flows_dir / "all_flows.csv", index=False)

### Split flows into fixed-size batches for API processing.

In [ ]:
BATCH_SIZE = 50 # Batch flows
output_file = "flow_algorithm_mapping.json"
batches = [flows.iloc[i:i+BATCH_SIZE] for i in range(0, len(flows), BATCH_SIZE)]
print(f'There are {len(batches)} batches of flows to process')

### Initialise or resume the flow-to-algorithm mapping from disk.

In [ ]:
output_json = {}
json_path = flows_dir / "flow_algorithm_mapping.json"

if json_path.exists():
    with open(json_path, "r") as f:
        output_json = json.load(f)

### Define the system prompt that instructs the LLM to classify OpenML flows into core algorithm families.

In [ ]:
system_prompt = """You are a machine learning expert specializing in analyzing OpenML flows. Your task is to classify flows into target algorithm categories or mark them as non-target.

**Target Algorithm Categories:**
1. **Support Vector Machine** - Support vector-based algorithms
2. **Decision Tree** - Single decision tree algorithms
3. **Linear Models** - Logistic regression, GLMNET, generalized linear models, ridge/lasso regression
4. **Random Forest** - Random forest ensemble methods
5. **XGBoost** - Extreme gradient boosting methods

**Output for target algorithms:** <flow_id>, <target_algorithm>, Classification  
**Output for non-target algorithms:** <flow_id>, NOT TARGET, Other

---

### Classification Approach:

**1. Pattern Recognition:** Look for algorithm names, abbreviations, and common implementations that clearly indicate one of the target algorithms.  
**2. Meta-Learning Models:** For ensemble or wrapper methods, identify the base learner. If the base learner is a target algorithm, classify accordingly.  
**3. Pipeline Processing:** For pipelines with preprocessing steps, identify the final predictive model. Classify based on that final model if it's a target algorithm.  
**4. Name Normalization:** Ignore common prefixes (library names), suffixes (version numbers), and implementation details to focus on the core algorithm.

---

### General Guidelines:

**Support Vector Machine indicators:**  
- Contains "SVM", "Support Vector", "SVC", "svm"  
- Vector-based classification methods  

**Decision Tree indicators:**  
- Contains "DecisionTree", "rpart", "J48", "C4.5", "CART", or "Tree"  
- Single tree methods (not ensembles)  

**Linear Models indicators:**  
- Contains "LogisticRegression", "GLMNET", "Linear", "Ridge", "Lasso"  
- Generalized linear regression variants  

**Random Forest indicators:**  
- Contains "RandomForest", "Random Forest", "ranger"  
- Tree-based ensembles with randomization  

**XGBoost indicators:**  
- Contains "XGBoost", "Extreme Gradient", "xgb"  
- Advanced gradient boosting implementations  

---

### Processing Rules:

1. Direct Algorithm Match: If flow name clearly indicates a target algorithm, classify it  
2. Base Learner Extraction: For meta-models, extract and evaluate the base algorithm  
3. Pipeline Final Model: For preprocessing pipelines, evaluate the final predictive component  
4. Conservative Classification: Only classify as target if clearly identifiable, otherwise use NOT TARGET  

---

### Output Format:
<flow_id>, <algorithm_name>, Classification  
OR  
<flow_id>, NOT TARGET, Other

**Requirements:**  
- Exactly 3 comma-separated values  
- Use exact names: "Support Vector Machine", "Decision Tree", "Linear Models", "Random Forest", "XGBoost"  
- Target algorithms use "Classification" type  
- Non-targets use "NOT TARGET, Other"  
- No explanations or additional text
"""

### Iteratively classify flows in batches via the LLM and persist results incrementally.

In [ ]:
for i, batch in enumerate(tqdm(batches, desc="Processing Batches")):  # Process all batches
    
    # Skip batch if all flow IDs are already processed
    if all(str(row["id"]) in output_json for _, row in batch.iterrows()):
        continue

    # Create fast lookup for flow names
    id_to_name = dict(zip(batch["id"].astype(str), batch["name"]))

    # Prepare user message
    flow_lines = "\n".join([f"{fid}, {name}" for fid, name in id_to_name.items()])
    user_prompt = f"Please process the following OpenML flows:\n\n{flow_lines}"

    try:
        response = client.chat.completions.create(
            # model="gpt-4",
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0
        )

        raw_reply = response.choices[0].message.content.strip()

        # Parse and store response - Updated for 3-part output
        for line in raw_reply.split("\n"):
            parts = [x.strip() for x in line.split(",", maxsplit=2)]  # Changed from maxsplit=3 to maxsplit=2
            if len(parts) == 3:  # Changed from 4 to 3
                fid, algorithm, alg_type = parts  # Updated unpacking
                if fid in id_to_name:
                    # Store with flow_id as key (which becomes index when converted to DataFrame)
                    output_json[fid] = {
                        "name": id_to_name[fid],
                        "algorithm_type": algorithm,  # Updated to match new prompt format
                        "classification_type": alg_type
                    }

    except Exception as e:
        pass  # Optional: log or retry

    # Save after every batch
    with open(json_path, "w") as f:
        json.dump(output_json, f, indent=2)

    time.sleep(1.0)  # Rate limiting

print("Done.")

### Filter out non-target flows and save the cleaned mapping.

In [ ]:
# Paths
input_path = flows_dir / "flow_algorithm_mapping.json"
output_path = flows_dir / "filtered_flow_algorithm_mapping.json"

# Load original JSON
with open(input_path, "r") as f:
    output_json = json.load(f)

# Filter out NOT TARGET entries
filtered_json = {
    fid: details
    for fid, details in output_json.items()
    if details.get("algorithm_type", "").strip().upper() != "NOT TARGET"
}

# Save filtered JSON
with open(output_path, "w") as f:
    json.dump(filtered_json, f, indent=2)

# Single concise log
print(
    f"Filtered {len(output_json)} → {len(filtered_json)} entries. "
    f"Saved → {output_path}"
)

### Inspect the set of unique algorithm families after filtering.

In [ ]:
with open(output_path, "r") as f:
    filtered_json = json.load(f)

# Collect unique algorithm types
unique_types = sorted({
    details["algorithm_type"].strip()
    for details in filtered_json.values()
    if details.get("algorithm_type")
})

# Single concise log
print(f"Algorithm types ({len(unique_types)}): " + ", ".join(unique_types))

### Identify and save flows whose algorithm labels fall outside the core set.

In [ ]:
core_algorithms = {
    "Decision Tree",
    "Random Forest",
    "Linear Models",
    "XGBoost",
    "Support Vector Machine",
}

# Load the filtered JSON
with open(output_path, "r") as f:
    data = json.load(f)

# Keep only non-core algorithms
mismatched_subset = {
    fid: entry
    for fid, entry in data.items()
    if entry.get("algorithm_type", "").strip() not in core_algorithms
}

# Save to flowey/
non_core_path = flows_dir / "non_core_algorithm_subset.json"
with open(non_core_path, "w") as f:
    json.dump(mismatched_subset, f, indent=2)

# Streamlined output
print(f"Non-core entries ({len(mismatched_subset)}). Saved → {non_core_path}")

### Prepare non-core flows for reclassification by batching and enabling resume support.

In [ ]:
# Load non-core entries (flowey/)
non_core_path = flows_dir / "non_core_algorithm_subset.json"
with open(non_core_path, "r") as f:
    bad_eggs = json.load(f)

# Convert to DataFrame for batching
bad_df = (
    pd.DataFrame.from_dict(bad_eggs, orient="index")
      .reset_index(names="id")
      .assign(id=lambda df: df["id"].astype(str))
)

# Batch
BATCH_SIZE = 5
bad_batches = [
    bad_df.iloc[i:i + BATCH_SIZE]
    for i in range(0, len(bad_df), BATCH_SIZE)
]

# Corrected output (flowey/)
corrected_path = flows_dir / "corrected_bad_eggs.json"
corrected_json = {}

# Resume if rerunning
if corrected_path.exists():
    with open(corrected_path, "r") as f:
        corrected_json = json.load(f)

### Reclassify non-core flows via the LLM, saving corrected labels incrementally.

In [ ]:
for i, batch in enumerate(tqdm(bad_batches, desc="Reprocessing non-core flows")):
    # Skip fully processed batches
    if all(row["id"] in corrected_json for _, row in batch.iterrows()):
        continue

    id_to_name = dict(zip(batch["id"], batch["name"]))
    flow_lines = "\n".join(f"{fid}, {name}" for fid, name in id_to_name.items())
    user_prompt = f"Please process the following OpenML flows:\n\n{flow_lines}"

    try:
        response = client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0,
        )

        raw_reply = response.choices[0].message.content.strip()

        for line in raw_reply.split("\n"):
            fid, algorithm, alg_type = map(str.strip, line.split(",", maxsplit=2))
            if fid in id_to_name:
                corrected_json[fid] = {
                    "name": id_to_name[fid],
                    "algorithm_type": algorithm,
                    "classification_type": alg_type,
                }

    except Exception as e:
        print(f"Batch {i} skipped ({e})")
        continue

    # Persist progress
    with open(corrected_path, "w") as f:
        json.dump(corrected_json, f, indent=2)

    time.sleep(1.0)

print(f"Reclassification complete. Total corrected: {len(corrected_json)}")

### Verify that all corrected classifications belong to the intended core algorithm families.

In [ ]:
# Load corrected non-core entries (flowey/)
with open(corrected_path, "r") as f:
    data = json.load(f)

# Collect unique algorithm types
unique_types = sorted({
    details["algorithm_type"].strip()
    for details in data.values()
    if details.get("algorithm_type")
})

core_set = {
    "Decision Tree",
    "Random Forest",
    "Linear Models",
    "Support Vector Machine",
    "XGBoost",
}

non_core = [alg for alg in unique_types if alg not in core_set]

# Single concise log
status = "OK" if not non_core else f"CHECK → {', '.join(non_core)}"
print(f"Corrected algorithm types ({len(unique_types)}): {', '.join(unique_types)} | {status}")

### Remove remaining non-target entries from corrected mappings and save the final filtered version.

In [ ]:
with open(corrected_path, "r") as f:
    data = json.load(f)

# Filter out NOT TARGET entries
filtered_data = {
    fid: entry
    for fid, entry in data.items()
    if entry.get("algorithm_type", "").strip().upper() != "NOT TARGET"
}

# Save filtered output (flowey/)
output_v2_path = flows_dir / "filtered_flow_algorithm_mapping_v2.json"
with open(output_v2_path, "w") as f:
    json.dump(filtered_data, f, indent=2)

# Single concise log
print(
    f"Filtered NOT TARGET: {len(data)} → {len(filtered_data)}. "
    f"Saved → {output_v2_path}"
)

### Merge corrected labels into the main mapping and write the final cleaned flow–algorithm mapping.

In [ ]:

main_path = flows_dir / "filtered_flow_algorithm_mapping.json"
corrections_path = flows_dir / "corrected_bad_eggs.json"
output_v2_path = flows_dir / "filtered_flow_algorithm_mapping_v2.json"

# Load main + corrections
with open(main_path, "r") as f:
    main_data = json.load(f)

with open(corrections_path, "r") as f:
    corrections = json.load(f)

# Merge corrections (overwrite by flow_id)
main_data.update(corrections)

# Remove NOT TARGET entries
cleaned_data = {
    fid: entry
    for fid, entry in main_data.items()
    if entry.get("algorithm_type", "").strip().upper() != "NOT TARGET"
}

# Save final cleaned mapping
with open(output_v2_path, "w") as f:
    json.dump(cleaned_data, f, indent=2)

# Single concise log
print(
    f"Merged + cleaned: {len(main_data)} → {len(cleaned_data)}. "
    f"Saved → {output_v2_path}"
)

### Perform a final sanity check on the algorithm families present in the cleaned mapping.

In [ ]:
# Load final mapping (flowey/)
final_path = flows_dir / "filtered_flow_algorithm_mapping_v2.json"
with open(final_path, "r") as f:
    filtered_json = json.load(f)

# Collect unique algorithm types
unique_types = sorted({
    details["algorithm_type"].strip()
    for details in filtered_json.values()
    if details.get("algorithm_type")
})

# Single concise log
print(f"Algorithm types ({len(unique_types)}): " + ", ".join(unique_types))